## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### `cross` vs. comma-syntax — when they differ
#### AMPL Formulation

In [11]:
%%ampl_eval
reset;      # clear previous model/data

set NODES;                          # nodes in a graph
set ARCS {NODES};                   # ARCS[i] = set of nodes reachable from node i

# Correct: comma-syntax allows ARCS[i] to depend on the first dummy index i
param cap {i in NODES, j in ARCS[i]} >= 0;  # capacity of arc (i,j)
var Flow {i in NODES, j in ARCS[i]} >= 0, <= cap[i,j];

# 'cross' cannot express this: NODES cross ARCS is not valid because
# ARCS is an indexed collection of sets, not a single set.
# The following would be WRONG:
# set ALL_ARCS = NODES cross ARCS;   # ERROR: ARCS is not a set expression

# The correct named-set equivalent uses setof:
set ALL_ARCS = setof {i in NODES, j in ARCS[i]} (i,j);  # 2-D set of valid pairs

In [12]:
%%ampl_eval
data;                   # Start data section

# Nodes in the graph
set NODES := A B C D;

# Adjacency structure: ARCS[i] = nodes reachable from i
set ARCS[A] := B C;
set ARCS[B] := C D;
set ARCS[C] := D;
set ARCS[D] := ;

# Arc capacities (i, j pairs consistent with ARCS)
param cap :=
A B 10
A C 5
B C 8
B D 7
C D 6;

display ALL_ARCS;

set ALL_ARCS := (A,B) (A,C) (B,C) (B,D) (C,D);



#### amplpy Formulation

In [14]:
# Nodes
ampl.set['NODES'] = ['A', 'B', 'C', 'D']

# Adjacency list (ARCS[i])
ampl.set['ARCS'] = {
    'A': ['B', 'C'],
    'B': ['C', 'D'],
    'C': ['D'],
    'D': []
}

# Arc capacities (sparse dictionary form)
ampl.param['cap'].set_values({
    ('A','B'): 10,
    ('A','C'): 5,
    ('B','C'): 8,
    ('B','D'): 7,
    ('C','D'): 6})

ARCS = ampl.param['cap'].get_values()

ALL_ARCS = [(r[0], r[1]) for r in ARCS]

print("ALL_ARCS =", ALL_ARCS)

ALL_ARCS = [('A', 'B'), ('A', 'C'), ('B', 'C'), ('B', 'D'), ('C', 'D')]
